In [89]:
import re
import uuid

In [90]:
def clean_phone(input):

    # Phone regex matches anything of the form ddd-ddd-dddd, where d is a digit [0-9]
    phone_re = r'\d{3}-\d{3}-\d{4}'

    # Return input without phone numbers
    return re.sub(phone_re, '*phone*', input)

In [91]:
def clean_date(input):

    # Date regex matches any valid dob (20th or 21st century)
    date_re = r'\b(0[1-9]|1[0-2])/(0[1-9]|[12][0-9]|3[01])/(19[0-9]{2}|20[0-9]{2})\b'

    # Return input without valid dates of birth
    return re.sub(date_re, '*date*', input)

In [92]:
def clean_dob(input):

    # Date regex matches any valid dob (20th or 21st century)
    date_re = r'\b(0[1-9]|1[0-2])/(0[1-9]|[12][0-9]|3[01])/(19[0-9]{2}|20[0-9]{2})\b'

    # The following tags identify a date of birth in the input
    tags = [r'dob', r'date of birth']
    for tag in tags:

        '''
        # Find all dates of birth for each tag
        matches = re.finditer(tag, input, re.IGNORECASE)
        for match in matches:
            i = match.start()

            # Substitute the first date after the tag, ensuring only a dob is censored
            # This will allow the program to ignore any dates that are not a dob
            input = input[:i] + re.sub(date_re, '*dob*', input[i:], count=1)
        '''
        input = re.sub(tag + r': ' + date_re, tag + r': ' + '*dob*', input, flags=re.IGNORECASE)
        
    # Return input without dates of birth
    return input

In [93]:
def clean_email(input):

    # Email regex matches any valid email address
    email_re = r'\b[a-zA-Z0-9][a-zA-Z0-9._-]*@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'

    # Return input without emails
    return re.sub(email_re, '*email*', input)

In [94]:
def clean_beneficiary_number(input):

    num_re = r'\d{3}-\d{4}-\d{4}'
    
    return re.sub(num_re, '*health plan beneficiary number*', input)

In [95]:
def clean_record_number(input):
    
    return clean_tag(input, 'medical record number')

def clean_certificate(input):

    return clean_tag(input, 'certificate number')

def clean_license(input):

    return clean_tag(input, 'license number')

def clean_serial(input):

    return clean_tag(input, 'pacemaker serial numbers')

def clean_identifier(input):

    return clean_tag(input, 'device identifier')

def clean_url(input):

    return clean_tag(input, 'url')

def clean_code(input):

    input = clean_tag(input, 'code')
    input = clean_tag(input, 'group no.')
    return clean_tag(input, 'health insurance')

In [96]:
def clean_tag(input, tag):
    
    # Find line identified as a tag
    index = re.search(tag + ':', input, re.IGNORECASE)
    if index is None:
        return input

    index = index.start()
    line_index = index + len(tag) + 1
    end = input.find('\n', line_index)

    # Return input without tag
    input = input.replace(input[line_index:end], ' *' + tag + '*')
    newline = input.find('\n', line_index)+1
    return input[:newline] + clean_tag(input[newline:], tag)

In [97]:
def clean_names(input):
    
    # Build list of names to anonymize
    names = []
    
    # Tags identify names that must be removed
    tags = [r'patient:', r'provider:', r'patient name:', r'provider name:', r'hospital name:', r'social worker:']
    removal_tags = [r'patient:', r'patient name:']
    # Alternative method of removing tagged names
    '''
    for tag in tags:
        
        matches = re.finditer(tag, input, re.IGNORECASE)
        for match in matches:
            i = match.start()
            end = input.find('\n', i)

            # Remove instances of identified names
            input = input[:i+len(tag)+1] + '*name*' + input[end:]
    '''

    # Line split approach
    lines = input.split('\n')
    for i in range(len(lines)):
        for tag in tags:
            if re.search(tag, lines[i], flags=re.IGNORECASE) is not None:

                # Add names to list for further anonymization
                if tag in removal_tags:
                    names = names + lines[i][len(tag) + 1:].split()
                
                lines[i] = lines[i][:len(tag)] + ' *' + tag[:-1] + '*'
                break
    input = '\n'.join(lines)

    # Now remove the patient's name throughout rest of document
    for name in names:
        input = re.sub(name, r'*name*', input)

    # Finally remove any honorifics that could give away information
    honorifics = [r'\bmr\.? ', r'\bmrs\.? ', r'\bdr\.? ', r'\bms\.? ', r'\bmiss\b']
    for honor in honorifics:
        input = re.sub(honor, '', input, flags=re.IGNORECASE)

    # Return input without names
    return input

In [98]:
def clean_address(input):

    # Find line identified as an address
    index = input.find('Address:')
    if index < 0:
        return input
    
    address_line_index = index + 9
    end = input.find('\n', address_line_index)

    # Return input without addresses
    input = input.replace(input[address_line_index:end], ' *address*')
    newline = input.find('\n', address_line_index)+1
    return input[:newline] + clean_address(input[newline:])

In [99]:
def clean_ssn(input):

    # Find line identified as an ssn
    index = input.find('SSN:')
    if index < 0:
        return input
    
    ssn_line_index = index + 5
    end = input.find('\n', ssn_line_index)

    # Return input without ssns
    input = input.replace(input[ssn_line_index:end], ' *SSN*')
    newline = input.find('\n', ssn_line_index)+1
    return input[:newline] + clean_ssn(input[newline:])

In [100]:
def clean_medicaid_acct(input):

    # Medicaid account regex matches nums of form xxxx xxxx xxxx xxxx
    acct_re = r'\d{4} \d{4} \d{4} \d{4}'

    input = re.sub(acct_re, '*medicaid account*', input, flags=re.IGNORECASE)

    # Return input without medicaid accounts
    return input

In [101]:
def clean_allergies(input, allergy_list):
    
    lines = input.split('\n')
    for i in range(len(lines)):
        for allergy in allergy_list:
            match = re.search(allergy, lines[i], flags=re.IGNORECASE)
            if match is not None:
                lines[i] = lines[i][:match.start()] + ' *allergy*'
                break
    input = '\n'.join(lines)

    return input

In [102]:
def clean_results(input):
    
    lines = input.split('\n')
    for i in range(len(lines)):
        match = re.search(r'lab results', lines[i], flags=re.IGNORECASE)
        if match is not None:
            lines[i] = lines[i][:match.start()] + ' *lab results*'
            i += 1
            while True:
                match = re.search(r'- ', lines[i])
                if match is not None:
                    lines[i] = ' - *lab result*'
                    i += 1
                else:
                    break
    input = '\n'.join(lines)

    return input

In [103]:
data_filenames = ['ehrJMS.txt', 'ehrMH2.txt', 'ehrEC3.txt']
anon_filenames = ['anondata_1.txt', 'anondata_2.txt', 'anondata_3.txt']
filter_filenames = ['phi1.txt', 'phi2.txt', 'PHI3.txt']

data_text = [open(i, 'r').read() for i in data_filenames]
filter_text = [open(i, 'r').read() for i in filter_filenames]

def generate_uuids(filenames):
    uuids = {}
    for filename in filenames:
        uuids[uuid.uuid4()] = filename
    return uuids

uuids = generate_uuids(data_filenames)

In [104]:
def get_filters(input):

    # Dictionary of filters that can be applied to document
    filters = {
        'name':False,
        'phone':False,
        'dates':False,
        'dob':False,
        'email':False,
        'address':False,
        'ssn':False,
        'acct':False,
        'allergies':False,
        'results':False,
        'record_num':False,
        'beneficiary_num':False,
        'certificate':False,
        'license':False,
        'serial':False,
        'identifier':False,
        'url':False,
        'code':False,
        'allergy_list':[]
    }

    # Mapping of different tags to filters
    # Allows file that specifies filters to use different syntax
    alias_dict = {
        'dates':'dates',
        'date of birth':'dob',
        'social security number':'ssn',
        'name':'name',
        'phone':'phone',
        'email':'email',
        'electronic mail address':'email',
        'address':'address',
        'medicaid account':'acct',
        'account number':'acct',
        'allergies':'allergies',
        'lab results':'results',
        'medical record number':'record_num',
        'certificate':'certificate',
        'license':'license',
        'serial number':'serial',
        'device identifier':'identifier',
        'url':'url',
        'identifying number':'code',
        'health plan beneficiary number':'beneficiary_num'
    }

    # Scan input file and turn on filters accordingly
    for filter in alias_dict.keys():
        if re.search(filter, input, re.IGNORECASE):
            filters[alias_dict[filter]] = True

    if filters['allergies']:
        lines = input.split('\n')
        for line in lines:
            if re.search('allergies', line, re.IGNORECASE):
                start = line.find('(') + 1
                end = line.find(')')
                filters['allergy_list'] = line[start:end].split('; ')

    # Return filter dictionary for use in anonymizing data
    return filters

filters = [get_filters(i) for i in filter_text]

In [105]:
def clean_text(input, filters):

    # Call various anonymizing functions depending on what filters are turned on
    if filters['phone']:
        input = clean_phone(input)
    if filters['dates']:
        input = clean_date(input)
    if filters['dob']:
        input = clean_dob(input)
    if filters['email']:
        input = clean_email(input)
    if filters['name']:
        input = clean_names(input)
    if filters['address']:
        input = clean_address(input)
    if filters['ssn']:
        input = clean_ssn(input)
    if filters['acct']:
        input = clean_medicaid_acct(input)
    if filters['allergies']:
        input = clean_allergies(input, filters['allergy_list'])
    if filters['results']:
        input = clean_results(input)
    if filters['beneficiary_num']:
        input = clean_beneficiary_number(input)
    if filters['record_num']:
        input = clean_record_number(input)
    if filters['certificate']:
        input = clean_certificate(input)
    if filters['license']:
        input = clean_license(input)
    if filters['serial']:
        input = clean_serial(input)
    if filters['identifier']:
        input = clean_identifier(input)
    if filters['url']:
        input = clean_url(input)
    if filters['code']:
        input = clean_code(input)
    

    # Return anonymized text
    return input

def write_uuids(texts, uuids):
    keys = list(uuids.keys())

    for i in range(len(texts)):
        texts[i] = '\nUUID: ' + str(keys[i]) + '\n\n' + texts[i]
    
    return texts

def get_uuid(text):

    uuid_pattern = r'UUID:\s*([a-fA-F0-9\-]{36})'
    found = re.search(uuid_pattern, text)
    if found:
        return uuid.UUID(found.group(1))
    return None

def get_filename(anon_filename, uuids):

    text = open(anon_filename, 'r').read()
    id = get_uuid(text)

    if id is not None:
        return uuids[id]
    return None


anon_text = [clean_text(i, j) for i, j in zip(data_text, filters)]
anon_text = write_uuids(anon_text, uuids)

anon_files = [open(i, 'w') for i in anon_filenames]

for i, j in zip(anon_files, anon_text):

    # Write anonymized text to new file
    i.write(j)
    i.close()

    # Print anonymized text
    print(j)
    print('\n')


UUID: d089fada-99ae-40f9-a1ca-d7be87e8db27

Patient: *patient*
date of birth: *dob*
Medical Record Number: 1234567
Date of Visit: 10/26/2023
Address:  *address*
Phone: *phone*
email: *email*
 


Provider: *provider*

Chief Complaint: "Persistent cough and shortness of breath for the past week."

History of Present Illness (HPI):
*name* presents to the clinic with a one-week history of a dry, hacking cough and progressive shortness of breath. He reports the cough began insidiously, initially mild, but has worsened over the past three days. He denies fever, chills, or night sweats. He describes the shortness of breath as a feeling of tightness in his chest, exacerbated by exertion. He has noticed a decrease in his exercise tolerance. He denies any recent travel or known exposures to sick individuals. He has tried over-the-counter cough suppressants with minimal relief.

Past Medical History (PMH):

Asthma (diagnosed in childhood)
Seasonal allergies
No known surgical history
Medications:

In [106]:
for filename in anon_filenames:
    print (get_filename(filename, uuids))

ehrJMS.txt
ehrMH2.txt
ehrEC3.txt


In [20]:
# Legacy anonymizing function, do not use
def legacy_anonymize(data_text):
    filters = ['Address','Date of Birth','Phone','email']
    
    data_text_lines = data_text.split('\n')
    
    for filter in filters:
        for i in range(len(data_text_lines)):
            if data_text_lines[i].find(filter) > -1:
                data_text_lines[i] = filter + ': *' + filter + '*'
    
    anon_text = '\n'.join(data_text_lines)
    anon_file = open('anondata.txt', 'w')
    anon_file.write(anon_text)
    
    print(anon_text)